# 06 -- подготовка данных для LSTM

Здесь оставляю базовые дневные последовательности из первого LSTM-ноутбука и добавляю к ним табличные признаки из [05_Data-Modeling.ipynb](/notebooks/modeling/05_Data-Modeling.ipynb) **(верю, что там все хорошо)**.

Итоговый объект для модели:

- последовательность последних 90 календарных дней;
- 91 готовый snapshot-признак из [Prepared_data.parquet](/data/Prepared_data.parquet);
- 4 дополнительных календарных snapshot-признака;
- target -- GMV следующих 30 дней.

Тяжелые rolling/delta/ratio-признаки **не сохраняю отдельным огромным тензором**; считаю их в [07_LSTM.ipynb](/notebooks/modeling/07_LSTM.ipynb) на батче. Сразу несколько плюсов:
* не раздуваю [data/lstm](/data/lstm) в сколько-то раз,
* могу менять набор динамических признаков без повторного прохода по 30 млн строк.

### **Про данные:** что оставляю, что добавляю, что не храню

**Оставляю в последовательности:** исходные дневные `search`, `cat`, счетчики воронки, `gmv*` и `active`. Все не бинарные величины храню как `log1p` -- тяжелые хвосты.

**Добавляю отдельно:** 91 признак из ноутбука `05` -- RFM, окна 7/30/90 дней, velocity/trend, стабильность, микроворонки, сезонность и monetary profile. **Еще добавляю синус/косинус дня года и недели для самой cutoff-даты -- нововведение.**

**Не храню как отдельные поля на диске:** `has_*`, rolling mean, rolling activity rate, дневные ratio и первые разности. Они дешево восстанавливаются из базовой последовательности и считаются в модели на ходу.

**Не использую как признак:** `user_id`, `cutoff_date`, target-поля. `user_id` нужен только для выравнивания и submission, `cutoff_date` -- для split и календаря.

> Это был краткий экскурс по [LSTM_FEATURES_AND_DATA.md](/docs/LSTM_FEATURES_AND_DATA.md).

In [6]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


def find_project_root(start=None):  # ноутбук будет работать и из корня, и из /notebooks
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "train.parquet").exists():
            return candidate
    raise FileNotFoundError("Не найден data/train.parquet")


PROJECT_ROOT = find_project_root()
INPUT_PATH = PROJECT_ROOT / "data" / "train.parquet"
PREPARED_PATH = PROJECT_ROOT / "data" / "Prepared_data.parquet"
OUTPUT_DIR = PROJECT_ROOT / "data" / "lstm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEQ_LEN = 90
HORIZON = 30

LABELED_CUTOFFS = pd.to_datetime([
    "2025-04-19", "2025-05-19", "2025-06-18", "2025-07-18",
    "2025-08-17", "2025-09-16", "2025-10-16", "2025-11-15",
    "2025-12-15", "2026-01-14",
])
INFERENCE_CUTOFF = pd.Timestamp("2026-02-13")
ALL_CUTOFFS = [*LABELED_CUTOFFS, INFERENCE_CUTOFF]

# Храню только базовые дневные каналы. Остальное достраиваю в 07 на батче.
BASE_SEQUENCE_FEATURES = [
    "search", "cat", "searches",
    "search_to_cart", "search_to_ord", "cat_to_cart", "cat_to_ord",
    "to_cart", "to_ord", "gmv_search", "gmv_cat", "gmv",
    "active",
]
RAW_FEATURES = BASE_SEQUENCE_FEATURES[:-1]
BINARY_FEATURES = {"search", "cat"}
LOG_FEATURES = [name for name in RAW_FEATURES if name not in BINARY_FEATURES]  # см. EDA
RAW_COLUMNS = ["event_date", "user_id", *RAW_FEATURES]

CALENDAR_SEQUENCE_FEATURES = [
    "dow_sin", "dow_cos",
    "dom_sin", "dom_cos",
    "doy_sin", "doy_cos",
]

EXTRA_STATIC_FEATURES = [
    "cutoff_doy_sin", "cutoff_doy_cos",
    "cutoff_week_sin", "cutoff_week_cos",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("base sequence channels:", len(BASE_SEQUENCE_FEATURES))

PROJECT_ROOT: /Users/pinta/Dev/E-CUP-2026
base sequence channels: 13


## Базовые последовательности

In [7]:
# Нужен только для пользователей, которых уже можно показывать модели на конкретном cutoff.
first_seen = pd.read_parquet(INPUT_PATH, columns=["user_id", "event_date"], engine="pyarrow")
first_seen["event_date"] = pd.to_datetime(first_seen["event_date"])
first_seen = first_seen.groupby("user_id", sort=True)["event_date"].min()


def build_base_cutoff(cutoff):
    cutoff = pd.Timestamp(cutoff)
    cutoff_dir = OUTPUT_DIR / cutoff.strftime("%Y-%m-%d")
    cutoff_dir.mkdir(parents=True, exist_ok=True)

    x_path = cutoff_dir / "X.npy"
    users_path = cutoff_dir / "user_id.npy"
    y_path = cutoff_dir / "y.npy"

    if x_path.exists() and users_path.exists() and (cutoff == INFERENCE_CUTOFF or y_path.exists()):
        print(cutoff.date(), "-- base sequence already exists, reuse")
        return

    history_start = cutoff - pd.Timedelta(days=SEQ_LEN - 1)
    is_inference = cutoff == INFERENCE_CUTOFF
    read_end = cutoff if is_inference else cutoff + pd.Timedelta(days=HORIZON)

    users = first_seen.index[first_seen <= cutoff].to_numpy(dtype=np.int64)
    user_to_row = pd.Series(np.arange(len(users), dtype=np.int32), index=users)

    # Читаю только нужное окно и нужные колонки.
    frame = pd.read_parquet(
        INPUT_PATH,
        columns=RAW_COLUMNS,
        engine="pyarrow",
        filters=[
            ("event_date", ">=", history_start),
            ("event_date", "<=", read_end),
        ],
    )
    frame["event_date"] = pd.to_datetime(frame["event_date"])

    history = frame[frame["event_date"] <= cutoff].copy()
    user_index = history["user_id"].map(user_to_row).to_numpy()
    day_index = (history["event_date"] - history_start).dt.days.to_numpy()

    # float16 экономит диск, а в DataLoader конкретный батч все равно перевожу в float32.
    X = np.zeros((len(users), SEQ_LEN, len(BASE_SEQUENCE_FEATURES)), dtype=np.float16)

    for j, name in enumerate(RAW_FEATURES):
        values = history[name].to_numpy(dtype=np.float32)
        if name in LOG_FEATURES:
            values = np.log1p(values)
        X[user_index, day_index, j] = values.astype(np.float16)

    # active отделяет день без строки от реальной строки с нулевыми счетчиками.
    X[user_index, day_index, -1] = 1.0

    np.save(cutoff_dir / "X.npy", X)
    np.save(cutoff_dir / "user_id.npy", users)

    if not is_inference:
        future = frame[frame["event_date"] > cutoff]
        y = (
            future.groupby("user_id")["gmv"].sum()
            .reindex(users, fill_value=0.0)
            .to_numpy(dtype=np.float32)
        )
        np.save(cutoff_dir / "y.npy", y)

    del frame, history, X


for cutoff in ALL_CUTOFFS:
    build_base_cutoff(cutoff)

## Snapshot-признаки из [05_Data-Modeling.ipynb](/notebooks/modeling/05_Data-Modeling.ipynb)

Здесь не пересчитываю 91 табличный признак второй раз. Беру уже готовый [Prepared_data.parquet](/data/Prepared_data.parquet), выравниваю строки строго по `user_id` из LSTM snapshot и сохраняю `static.npy`.

К исходным 91 признакам добавляю 4 циклических признака cutoff. Для heavy-tail статических полей в [07_LSTM.ipynb](/notebooks/modeling/07_LSTM.ipynb) дополнительно создаю `log1p`-копии, а для всех `NaN` -- missing-mask. Это дешево и не требует менять [Prepared_data.parquet](/data/Prepared_data.parquet).

> Но при обновлении [Prepared_data.parquet](/data/Prepared_data.parquet) все нужно еще раз запустить!!!

In [8]:
prepared_ds = ds.dataset(PREPARED_PATH, format="parquet")
PREPARED_FEATURES = [
    name for name in prepared_ds.schema.names
    if name not in {"user_id", "cutoff_date", "target_gmv_30d", "target_nonzero"}
]

STATIC_FEATURES = [*PREPARED_FEATURES, *EXTRA_STATIC_FEATURES]


def static_log_copy_candidates(feature_names):
    # Добавляю log1p-копии только для неотрицательных magnitude/recency признаков.
    include = ("gmv", "searches", "items", "days", "customer_age", "gap")
    exclude = ("trend", "share", "rate", "ratio", "velocity", "cv", "sin", "cos", "score", "frequency")
    return [
        name for name in feature_names
        if any(token in name for token in include)
        and not any(token in name for token in exclude)
    ]


STATIC_LOG_COPY_FEATURES = static_log_copy_candidates(STATIC_FEATURES)


def cutoff_static_calendar(cutoff, n_rows):
    day_of_year = cutoff.dayofyear
    week_of_year = int(cutoff.isocalendar().week)
    values = np.array([
        np.sin(2 * np.pi * day_of_year / 365.25),
        np.cos(2 * np.pi * day_of_year / 365.25),
        np.sin(2 * np.pi * week_of_year / 52.0),
        np.cos(2 * np.pi * week_of_year / 52.0),
    ], dtype=np.float32)  # TODO можно поэкспериментировать и как-то еще календарные признаки затригонометрить
    return np.broadcast_to(values, (n_rows, len(values))).copy()


def sequence_calendar(cutoff):
    dates = pd.date_range(cutoff - pd.Timedelta(days=SEQ_LEN - 1), cutoff, freq="D")
    dow = dates.dayofweek.to_numpy()
    dom = dates.day.to_numpy()
    doy = dates.dayofyear.to_numpy()

    return np.column_stack([
        np.sin(2 * np.pi * dow / 7.0), np.cos(2 * np.pi * dow / 7.0),
        np.sin(2 * np.pi * (dom - 1) / 31.0), np.cos(2 * np.pi * (dom - 1) / 31.0),
        np.sin(2 * np.pi * doy / 365.25), np.cos(2 * np.pi * doy / 365.25),
    ]).astype(np.float32)


def build_static_cutoff(cutoff):
    cutoff = pd.Timestamp(cutoff)
    cutoff_dir = OUTPUT_DIR / cutoff.strftime("%Y-%m-%d")
    users = np.load(cutoff_dir / "user_id.npy", mmap_mode="r")

    table = prepared_ds.to_table(
        columns=["user_id", *PREPARED_FEATURES],
        filter=ds.field("cutoff_date") == cutoff.date(),
    )
    snapshot = table.to_pandas().set_index("user_id")
    snapshot = snapshot.reindex(np.asarray(users))

    static = snapshot[PREPARED_FEATURES].to_numpy(dtype=np.float32)
    static = np.concatenate([static, cutoff_static_calendar(cutoff, len(static))], axis=1)

    np.save(cutoff_dir / "static.npy", static.astype(np.float32))
    np.save(cutoff_dir / "calendar.npy", sequence_calendar(cutoff))

    print(cutoff.date(), "static:", static.shape, "calendar:", (SEQ_LEN, len(CALENDAR_SEQUENCE_FEATURES)))


for cutoff in ALL_CUTOFFS:
    build_static_cutoff(cutoff)

2025-04-19 static: (216457, 95) calendar: (90, 6)
2025-05-19 static: (221154, 95) calendar: (90, 6)
2025-06-18 static: (225245, 95) calendar: (90, 6)
2025-07-18 static: (229146, 95) calendar: (90, 6)
2025-08-17 static: (232977, 95) calendar: (90, 6)
2025-09-16 static: (236668, 95) calendar: (90, 6)
2025-10-16 static: (240700, 95) calendar: (90, 6)
2025-11-15 static: (244983, 95) calendar: (90, 6)
2025-12-15 static: (250000, 95) calendar: (90, 6)
2026-01-14 static: (250000, 95) calendar: (90, 6)
2026-02-13 static: (250000, 95) calendar: (90, 6)


## Итого признаки в [07_LSTM.ipynb](/notebooks/modeling/07_LSTM.ipynb):

На диске остаются 13 базовых sequence-каналов и 95 static-полей. Перед LSTM я дополнительно строю уже на ходу:

- 4 явных `has_*` индикатора;
- 4 дневных ratio по воронке и GMV;
- rolling mean за 7 и 30 дней для `searches`, `to_cart`, `to_ord`, `gmv`;
- rolling activity rate за 7 и 30 дней;
- первые разности для `searches`, `to_ord`, `gmv`;
- 6 календарных sequence-признаков;
- `log1p`-копии тяжелых static-признаков;
- missing-mask для каждого static-признака.

**Профит:** исходный тензор на диске остается компактным, но реальный вход модели заметно богаче.

In [9]:
meta = {
    "seq_len": SEQ_LEN,
    "horizon": HORIZON,
    "base_sequence_features": BASE_SEQUENCE_FEATURES,
    "calendar_sequence_features": CALENDAR_SEQUENCE_FEATURES,
    "prepared_static_features": PREPARED_FEATURES,
    "extra_static_features": EXTRA_STATIC_FEATURES,
    "static_features": STATIC_FEATURES,
    "static_log_copy_features": STATIC_LOG_COPY_FEATURES,
    "labeled_cutoffs": [x.strftime("%Y-%m-%d") for x in LABELED_CUTOFFS],
    "inference_cutoff": INFERENCE_CUTOFF.strftime("%Y-%m-%d"),
    "base_transform": "log1p for non-binary daily channels; active is 0/1",
    "target": "sum(gmv) for cutoff < event_date <= cutoff + 30 days",
}

with open(OUTPUT_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("saved meta:", OUTPUT_DIR / "meta.json")
print("static base features:", len(STATIC_FEATURES))
print("static log copies in 07:", len(STATIC_LOG_COPY_FEATURES))

saved meta: /Users/pinta/Dev/E-CUP-2026/data/lstm/meta.json
static base features: 95
static log copies in 07: 51
